In [0]:
# ═══════════════════════════════════════════════════════════════════════════════
# PIPELINE FILE: p2_silver_pipeline
# PURPOSE      : DLT Silver layer — clean, validate, type-cast, and model
#                raw Bronze data into a proper star schema.
#
# THIS IS NOT A REGULAR NOTEBOOK.
# Attach it to a new DLT pipeline called pipeline_silver in the UI.
#
# WHAT "SILVER" MEANS:
#   Silver = cleansed, typed, validated, business-meaningful data.
#   No raw strings where numbers belong. No nulls where values are required.
#   No duplicates. JSON columns exploded into proper relational columns.
#   Silver is what analysts and BI tools query — not Bronze.
#
# INPUTS (reads from Bronze streaming tables + Volume static CSVs):
#   bronze.orders                 → live POS orders stream
#   bronze.historical_orders_raw  → historical batch orders stream
#   bronze.review_raw             → customer reviews stream
#   /Volumes/.../restaurants/     → static CSV (5 rows)
#   /Volumes/.../customers/       → static CSV (500 rows)
#   /Volumes/.../menu_items/      → static CSV (~125 rows)
#
# OUTPUTS (Silver star schema):
#   silver.fact_orders      → unified fact table (live + historical orders)
#   silver.fact_reviews     → cleaned reviews with rating validation
#   silver.dim_restaurants  → restaurant dimension (static)
#   silver.dim_customers    → customer dimension (static)
#   silver.dim_menu_items   → menu item dimension (static)
#
# STAR SCHEMA EXPLAINED:
#   A star schema has one central FACT table (the measurable events — orders)
#   surrounded by DIMENSION tables (the descriptive context — who, where, what).
#   Gold layer will JOIN fact_orders WITH dimensions to answer business questions.
#   This is the same model used in Synapse Dedicated SQL Pool with PolyBase loads.
# ═══════════════════════════════════════════════════════════════════════════════

import dlt
from pyspark.sql.functions import (
    col, to_timestamp, trim, upper, lower, when, lit,
    from_json, explode, round as spark_round,
    current_timestamp, coalesce, regexp_replace
)
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType,
    IntegerType, ArrayType, TimestampType
)

CATALOG  = "restaurant_catalog"
VOL_PATH = f"/Volumes/{CATALOG}/landing/raw_files"

In [0]:
# ─── HELPER: JSON SCHEMA FOR items COLUMN ─────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   The Bronze `items` column contains raw JSON strings like:
#   '[{"item_id":"ITEM0001","item_name":"Butter Chicken","category":"Main Course",
#      "quantity":2,"unit_price":55.0,"subtotal":110.0}]'
#
#   To use this data in SQL queries (GROUP BY category, SUM subtotal),
#   you need to PARSE the JSON string into actual typed columns.
#   This schema tells Spark exactly what structure to expect inside the JSON.
#
# WHY DEFINE THE SCHEMA EXPLICITLY (not use schema_of_json):
#   schema_of_json() infers the schema from a sample value — it can guess
#   wrong types or miss nullable fields. Explicit schema = guaranteed types.
#   In production, JSON schema definitions are stored in a schema registry.
#   Here we define it inline — same principle, simpler setup.
#
# WHY ArrayType wrapping StructType:
#   The items column is a JSON ARRAY of objects: [{...}, {...}, {...}]
#   ArrayType = the outer [ ] brackets
#   StructType = the inner { } object structure
#   Each order can have 1 to 4 items — hence ArrayType, not just StructType.
# ──────────────────────────────────────────────────────────────────────────────

ITEMS_SCHEMA = ArrayType(StructType([
    StructField("item_id",    StringType(),  True),
    StructField("item_name",  StringType(),  True),
    StructField("category",   StringType(),  True),
    StructField("quantity",   IntegerType(), True),
    StructField("unit_price", DoubleType(),  True),
    StructField("subtotal",   DoubleType(),  True),
]))

In [0]:
# ─── TABLE 1: silver.fact_orders ─────────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   This is the most important table in the entire project.
#   You are combining TWO Bronze sources into ONE unified fact table:
#     Source A: bronze.orders       → live POS orders (25 rows, grows daily)
#     Source B: bronze.historical_orders_raw → 6-month history (8,000 rows)
#
#   Why combine them? Because analysts don't care WHERE the data came from —
#   they want to query ALL orders in one place across all time.
#
# HOW dlt.read_stream() WORKS:
#   dlt.read_stream("orders") = read from bronze.orders AS A STREAM
#   This is the DLT-native way to read from another DLT table.
#   Do NOT use spark.readStream.table("restaurant_catalog.bronze.orders") here —
#   that bypasses DLT's dependency tracking and lineage graph.
#   Using dlt.read_stream() tells DLT: "silver.fact_orders DEPENDS ON bronze.orders"
#   DLT draws the dependency arrow in the UI pipeline DAG automatically.
#
# WHY UNION instead of JOIN:
#   Both sources have the SAME columns (order_id, timestamp, restaurant_id, etc.)
#   UNION stacks them vertically — row from source A + row from source B.
#   JOIN combines columns horizontally — used when sources have DIFFERENT columns.
#   Think: UNION = adding more rows. JOIN = adding more columns.
#
# WHY dropDuplicates(["order_id"]):
#   Safety net. If for any reason the same order_id arrives in BOTH sources
#   (e.g., a historical order that was also captured as a live order),
#   this removes the duplicate. In production: always deduplicate on business key.
#
# WHY to_timestamp():
#   Bronze stores timestamp as a STRING "2026-04-12 14:30:00".
#   Silver converts it to TimestampType — a proper datetime value.
#   This enables: date_trunc(), year(), month(), dayofweek() in Gold aggregations.
#   Without this cast, you cannot do time-based analytics.
#
# @dlt.expect_or_drop:
#   This is a DATA QUALITY RULE. If the condition is FALSE, the row is DROPPED.
#   "valid_order_id"    → order_id cannot be null (every order must have an ID)
#   "valid_total"       → total_amount must be > 0 (zero-value orders are invalid)
#   "valid_restaurant"  → restaurant_id must be one of our 5 known restaurants
#   Dropped rows are logged in the DLT pipeline UI under "Dropped" column.
#   You saw "Dropped: 0" in your Bronze run — that means all rows passed.
#   expect_or_drop = drop bad rows silently (use for data quality enforcement)
#   expect_or_fail = fail the entire pipeline if any row fails (use for critical rules)
# ──────────────────────────────────────────────────────────────────────────────

@dlt.table(
    name = "fact_orders",
    comment = "Silver: unified fact table - live POS orders UNION 6-month historical orders",
    table_properties = {
        "quality" : "silver",
        "pipeline.reset.allowed":"false",
    }
)
@dlt.expect_or_drop("valid_order_id","order_id IS NOT NULL")
@dlt.expect_or_drop("valid_total","total_amount > 0")
@dlt.expect_or_drop("valid_restaurant","restaurant_id IN ('R001','R002','R003','R004','R005')")
@dlt.expect_or_drop("valid_status","order_status IN ('completed','pending','cancelled')")
def fact_orders():
    # read live orders from bronze table
    live_orders = (
        dlt.read_stream("restaurant_catalog.bronze.orders")
        .select(
            col("order_id"),
            to_timestamp(col("timestamp"),"yyyy-MM-dd HH:mm:ss").alias("order_timestamp"),
            col("restaurant_id"),
            col("customer_id"),
            col("items"),
            col("total_amount").cast(DoubleType()),
            col("payment_method"),
            col("order_type"),
            col("order_status"),
            lit("live").alias("source"),
        )
    )

    # read historical order

    hist_orders = (
        dlt.read_stream("restaurant_catalog.bronze.historical_orders_raw")
        .select(
            col("order_id"),
            to_timestamp(col("timestamp"),"yyyy-MM-dd HH:mm:ss").alias("order_timestamp"),
            col("restaurant_id"),
            col("customer_id"),
            col("items"),
            col("total_amount").cast(DoubleType()),
            col("payment_method"),
            col("order_type"),
            col("order_status"),
            lit("historical").alias("source"),
        )
    )

        # union
    return (live_orders.unionByName(hist_orders).dropDuplicates(["order_id"]))


In [0]:
# ─── TABLE 2: silver.fact_reviews ─────────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Cleaning the raw reviews from Bronze. Key transformations:
#   1. Cast rating to IntegerType (CSV may have read it as string)
#   2. Validate rating is between 1 and 5 — drop anything outside this range
#   3. Trim whitespace from text fields (common CSV quality issue)
#   4. Add review_date as proper DateType
#
# WHY @dlt.expect_or_drop on rating:
#   Rating is the core metric in the Gold layer (avg_rating per restaurant).
#   A rating of 0, 6, or null would corrupt every downstream aggregation.
#   Dropping at Silver ensures Gold aggregations are always on valid data.
#   This is the "quality gate" pattern — bad data is stopped at Silver,
#   never allowed to contaminate Gold.
#
# NOTE ON dlt.read_stream("review_raw"):
#   Your Bronze table is named "review_raw" (not "reviews_raw").
#   This matches exactly what you committed in p1_bronze_pipeline.
#   Always match the @dlt.table name= parameter exactly.
# ──────────────────────────────────────────────────────────────────────────────

@dlt.table(
    name    = "fact_reviews",
    comment = "Silver: cleaned customer reviews. Rating validated (1-5). "
              "Text trimmed. Date cast to DateType. "
              "Source: bronze.review_raw via Auto Loader.",
    table_properties = {"quality": "silver"}
)
@dlt.expect_or_drop("valid_rating",  "rating BETWEEN 1 AND 5")
@dlt.expect_or_drop("valid_review_id", "review_id IS NOT NULL")
@dlt.expect_or_drop("valid_order_ref", "order_id IS NOT NULL")
def fact_reviews():
    return (
        dlt.read_stream("restaurant_catalog.bronze.review_raw")
           .select(
               col("review_id"),
               col("order_id"),
               col("customer_id"),
               col("restaurant_id"),
               col("rating").cast(IntegerType()),          # CSV may read as string — cast to int
               trim(col("review_text")).alias("review_text"),  # remove leading/trailing spaces
               col("review_date").cast("date").alias("review_date"),  # string → DateType
           )
    )

In [0]:
# ─── TABLE 3: silver.dim_restaurants ─────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Creating the Restaurant DIMENSION table from the static CSV in the Volume.
#   Dimension tables describe the "who/what/where" context of fact table events.
#   dim_restaurants answers: "Tell me about restaurant R001"
#
# WHY spark.read (NOT dlt.read_stream or readStream):
#   There are only 5 restaurants. This data NEVER changes during the project.
#   Using readStream for 5 static rows would be wasteful — it would set up
#   a streaming query with checkpoint for data that never updates.
#   spark.read = read once, create a Materialized View in DLT.
#   DLT will recreate this table on every full pipeline refresh.
#
# WHY read_csv_from_volume() HELPER:
#   Same pattern as in 02_order_producer — Spark writes CSVs to a folder
#   with part files. We need to find the actual .csv file dynamically.
#   In Silver, we use spark.read instead of pandas because we return
#   a Spark DataFrame to DLT (DLT cannot accept a Pandas DataFrame).
#
# TRANSFORMATION:
#   trim() on name and location — removes any whitespace from CSV parsing.
#   upper(city) — standardizes city names: "dubai" and "Dubai" → "DUBAI"
#   This prevents duplicate groupings in Gold: GROUP BY city would treat
#   "Dubai" and "dubai" as different cities without this normalization.
# ──────────────────────────────────────────────────────────────────────────────

def read_volume_csv_spark(folder_name:str):
    folder_path = f"{VOL_PATH}/{folder_name}"
    all_files = dbutils.fs.ls(folder_path)

    csv_file = [item.path for item in all_files if item.name.endswith(".csv")][0]

    return (
        spark.read.option("header","true")
        .option("inferSchema","true")
        .csv(csv_file)
    )


@dlt.table(
    name="dim_restaurants",
    comment="Silver: Restaurants",
    table_properties = {
        "quality":"Silver"
    }
)
def dim_restaurants():
    return (
        read_volume_csv_spark("restaurants")
        .select(
            col("restaurant_id"),
            trim(col("name")).alias("restaurant_name"),
            upper(col("city")).alias("city"),
            trim(col("location")).alias("location"),
            col("cuisine"),
        )
    )


In [0]:
# ─── TABLE 4: silver.dim_customers ────────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Creating the Customer DIMENSION table from the static customers CSV.
#   500 customer records with loyalty tier, city, and join date.
#
# KEY TRANSFORMATION — loyalty_tier standardization:
#   Raw CSV may have inconsistent casing: "bronze", "Bronze", "BRONZE"
#   upper() normalizes all to: "BRONZE", "SILVER", "GOLD", "PLATINUM"
#   This matters in Gold: GROUP BY loyalty_tier would create 3 groups
#   for the same tier without normalization.
#
# WHY join_date AS DateType:
#   Gold layer needs to calculate customer tenure:
#   datediff(current_date(), join_date) = days since customer joined
#   This only works if join_date is a proper DateType, not a string.
#
# INTERVIEW CONTEXT — SCD Type 2:
#   In production, the customer dimension would be an SCD Type 2 table —
#   tracking history of changes (e.g., loyalty_tier upgrades over time).
#   For this project, we implement a static dimension (SCD Type 1 effectively).
#   In an interview: "I implemented dim_customers as a static Silver dimension.
#   In production I'd add SCD Type 2 using DLT's APPLY CHANGES INTO feature,
#   which tracks is_current, valid_from, valid_to columns automatically."
# ──────────────────────────────────────────────────────────────────────────────

@dlt.table(
    name="dim_customers",
    comment="Silver: Customer Dimension table as of now it's SCD Type 1",
    table_properties = {"quality":"silver"}
)
def dim_customers():
    return (
        read_volume_csv_spark("customers")
        .select(
            col("customer_id"),
            trim(col("name")).alias("customer_name"),
            lower(col("email")).alias("email"),
            col("phone"),
            upper(col("city")).alias("city"),
            upper(col("loyalty_tier")).alias("loyalty_tier"),
            col("join_date").cast("date").alias("joine_date"),
        )
    )

In [0]:
# ─── TABLE 5: silver.dim_menu_items ───────────────────────────────────────────
#
# WHAT YOU'RE DOING:
#   Creating the Menu Item DIMENSION table from the menu_items CSV.
#   ~125 rows (25 items × 5 restaurants).
#   This dimension will be used in Gold to answer:
#   "Which category (Main Course / Starter / Dessert) generates the most revenue?"
#
# KEY TRANSFORMATION — is_available:
#   CSV stores booleans as strings: "True" / "False"
#   We convert to a proper BooleanType column.
#   when(col("is_available") == "True", True).otherwise(False)
#   This enables: WHERE is_available = true in Gold queries.
#
# WHY unit_price AS DoubleType:
#   Arithmetic on a StringType column raises an AnalysisException.
#   Casting to DoubleType enables: SUM(unit_price), AVG(unit_price) in Gold.
# ──────────────────────────────────────────────────────────────────────────────

@dlt.table(
    name = "dim_menu_items",
    comment="Silver: Menu Items",
    table_properties = {
        "quality":"silver"
    }
)
def dim_menu_items():
    return (
        read_volume_csv_spark("menu_items")
        .select(
            col("item_id"),
            col("restaurant_id"),
            trim(col("item_name")).alias("item_name"),
            col("category"),
            col("unit_price").cast(DoubleType()),
            when(col("is_available")=="True",True)
            .otherwise(False)
            .alias("is_available"),
        )
    )